# 第7回　推定：点推定と区間推定
## ―― 「信頼区間95%」の、本当の意味

統計学Ⅰ（B）　／　北星学園大学

第6回で「標本平均は母平均の周りに SE=σ/√n で散らばる」と分かった。今日はそれを使って母平均を**区間**で推定する。注目は ――

> 「95%信頼区間」の95%は、**あなたが思っている意味とは、たぶん違う。**

### まず、自分の答えを書いてみよう

> **「95%信頼区間」とはどういう意味だと思いますか？**
> 
> 多くの人はこう答える：「真の値が、95%の確率でこの区間の中にある」。
> 
> ――この答え、実は**誤り**だ。なぜかを、今日シミュレーションで確かめる。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 0. 今日の母集団

第6回と同じ、**体重の記録がある霊長類265種**を母集団とする。

In [ ]:
rng = np.random.default_rng(2026)

母集団 = df["体重g"].dropna().values
mu    = 母集団.mean()
sigma = 母集団.std()
n     = 30
SE    = sigma / np.sqrt(n)

print(f"母平均 μ　　　 = {mu:,.0f} g")
print(f"母標準偏差 σ　 = {sigma:,.0f} g")
print(f"標本サイズ n　 = {n} 種")
print(f"標準誤差 SE　  = σ/√n = {SE:,.0f} g")

---
## 1. 点推定 ―― 1つの値で当てる

標本平均を、母平均の**推定値**として使う。これが**点推定**。でも標本ごとに値は揺れ、母平均にぴったりは当たらない。

In [ ]:
for i in range(5):
    xbar = rng.choice(母集団, n).mean()
    print(f"標本{i+1} の点推定（標本平均）= {xbar:>7,.0f} g　（母平均 {mu:,.0f} とのズレ {xbar-mu:>+7,.0f}）")
print("\n点推定は『当たり』を言い切るが、必ず少しズレる。どれくらい外れうるか＝幅が欲しい。")

---
## 2. 区間推定 ―― 幅を持たせる

そこで「だいたいこの範囲」と幅で示す。標本平均の散らばりは SE。正規分布では95%が平均±1.96SDに入るので、

$$ 95\%信頼区間 = \bar{x} \pm 1.96 \times SE $$

In [ ]:
xbar = rng.choice(母集団, n).mean()
half = 1.96 * SE
print(f"ある標本の平均 x̄ = {xbar:,.0f} g")
print(f"95%信頼区間 = [{xbar-half:,.0f}, {xbar+half:,.0f}]（半幅 {half:,.0f}）")
print(f"→ この区間は母平均 {mu:,.0f} を含んでいる")

---
## 3. では「95%」とは何の確率か？

ここが核心。多くの人は「**真の母平均が、95%の確率でこの区間に入る**」と思う。だが ――

**母平均 μ は固定された定数**だ（5,881gで動かない）。区間も計算してしまえば固定。固定した値が固定した区間に「95%の確率で入る」は意味をなさない。入っているか・いないか、どちらかでしかない。

では95%は何か。**ランダムなのは『区間のほう』**だ。標本を取り直すたびに区間は動く。**その手続きを何度も繰り返せば、作った区間の約95%が母平均を含む。** ―― これが95%の正体。確かめよう。

In [ ]:
plt.figure(figsize=(7, 8))
含んだ = 0
for i in range(100):
    xbar = rng.choice(母集団, n).mean()
    lo, hi = xbar - 1.96*SE, xbar + 1.96*SE
    当たり = lo <= mu <= hi
    含んだ += 当たり
    plt.plot([lo, hi], [i, i], color="#00897b" if 当たり else "#e8503a", lw=1.5, alpha=0.8)
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 μ={mu:,.0f}g（真の値・固定）")
plt.xlabel("体重（g）"); plt.ylabel("標本の番号（1〜100）")
plt.title(f"100本の95%信頼区間：母平均を含んだのは {含んだ} 本（赤＝外した）")
plt.legend(loc="lower right"); plt.show()
print(f"100本中 {含んだ} 本が母平均を含んだ → これが『95%』の意味")

**100本中およそ95本**が母平均（青線・固定）を含み、約5本（赤）は外した。

「95%」とは、**この区間作りを繰り返したときの“当たり率”**であって、「いま手元の1本に真値が95%で入っている」ではない。手元の1本は、当たっているか外しているかのどちらかだ（どちらかは分からないが）。

In [ ]:
# もっと大きい回数で「当たり率」を確かめる
試行 = 20000
xb = np.array([rng.choice(母集団, n).mean() for _ in range(試行)])
カバー率 = np.mean((xb - 1.96*SE <= mu) & (mu <= xb + 1.96*SE)) * 100
print(f"{試行:,}回くりかえしたときの当たり率 = {カバー率:.1f}%")
print("→ 設計どおり、ほぼ95%。区間の作り方としては正しく機能している。")

> **第6回とのつながり。**
> 第6回で「この母集団は歪度7.60で、n=30では標本平均の分布がまだ正規に届かない」と確かめた。
> それなのにカバー率はきちんと95%になっている。**区間は左右対称なので、片側で多く外し、もう片側で少なく外して、結果的に釣り合うから**である。
> 
> **「うまくいっている」ことと「前提が満たされている」ことは別**だ。次の節でその代償を見る。

---
## 4. 信頼度と幅のトレードオフ

「もっと確実に当てたい」なら信頼度を上げる（95%→99%）。すると**区間は広がる**。確実さと、区間の狭さ（情報の精密さ）は両立しない。

In [ ]:
for 信頼度, z in [("90%", 1.645), ("95%", 1.96), ("99%", 2.576)]:
    print(f"{信頼度}信頼区間の半幅 = {z}×SE = {z*SE:>6,.0f} g")
print("\n確実にしたい（99%）ほど区間は広く＝ぼんやりする。")
print("極端に『100%確実』にすると幅は無限大（『体重は0〜∞g』）＝何も言っていないのと同じ。")

---
## 5. その区間、ありえない値を含んでいないか

カバー率は95%で正しかった。では、この区間をそのまま報告してよいのか。**実際に作られる区間を、いくつか見てみよう。**

In [ ]:
print("5つの標本から作った95%信頼区間:")
for i in range(5):
    xb1 = rng.choice(母集団, n).mean()
    lo, hi = xb1 - 1.96*SE, xb1 + 1.96*SE
    印 = "  ← 下端が負！" if lo < 0 else ""
    print(f"  標本{i+1}: x̄={xb1:>7,.0f}g → [{lo:>9,.0f}, {hi:>8,.0f}]{印}")

# 何割の区間が「負の下端」を持つか
負の割合 = np.mean(xb - 1.96*SE < 0) * 100
print(f"\n下端が負になる区間の割合 = {負の割合:.1f}%")

**約39%の区間が「体重がマイナス」を含んでいる。**

「霊長類の平均体重は **−326g 〜 9,056g** です」―― これは報告として成立しない。**体重がマイナスの動物はいない。**

第3回で「生のスケールに正規分布を当てると平均−1SDが負になる」と見た。同じことが区間でも起きている。カバー率という数字だけを見ていては、この欠陥に気づけない。

**対数スケールで同じことをすると、どうなるか。**

In [ ]:
対数母集団 = np.log10(母集団)
lmu = 対数母集団.mean(); lSE = 対数母集団.std() / np.sqrt(n)

print("対数スケールで区間を作り、元の単位（g）に戻したもの:")
for i in range(5):
    xb2 = rng.choice(対数母集団, n).mean()
    lo, hi = xb2 - 1.96*lSE, xb2 + 1.96*lSE
    print(f"  標本{i+1}: [{10**lo:>8,.0f}g, {10**hi:>8,.0f}g]   ← 必ず正")

# 対数版のカバー率
xb3 = np.array([rng.choice(対数母集団, n).mean() for _ in range(20000)])
print(f"\n対数スケールでのカバー率 = "
      f"{np.mean((xb3-1.96*lSE <= lmu) & (lmu <= xb3+1.96*lSE))*100:.1f}%")

**対数スケールなら、区間は必ず正の値になる。** カバー率も95%を保っている。

> ただし注意。対数スケールで作った区間を戻したものは、**幾何平均の信頼区間**であって、算術平均のそれではない。**何を推定したいのかで、選ぶ尺度が変わる。**

> **今日の隠れた教訓。**
> 「カバー率95%」という数字は正しかった。それでも、その区間は**言ってはいけないこと**を言っていた。
> **指標が合格していても、出力そのものを見る。**

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 点推定 | 標本平均1つで母平均を当てる。必ず少しズレる |
| 区間推定 | x̄ ± 1.96×SE で「だいたいこの範囲」 |
| 95%の意味 | 手続きを繰り返すと**約95%の区間が真値を含む**（当たり率） |
| よくある誤り | 「真値が95%でこの区間に入る」「μが確率的に動く」 |
| 信頼度と幅 | 確実にするほど区間は広がる（トレードオフ） |
| **区間の中身を見る** | **カバー率が正しくても、ありえない値を含むことがある**（39%が負の下端） |

> **母平均は固定。ランダムなのは区間のほう。**
> 「95%」は、この区間作りを100回やれば約95回当たる、という手続きの当たり率だ。

次回からは「差があるか」を判定する**仮説検定**へ。

**課題（Moodle）**：「95%信頼区間」のよくある誤った説明文の、どこが誤りかを正す。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。